<a href="https://colab.research.google.com/github/emanhassan2020/HandsOn/blob/main/Deployment/export_control_flow_model_to_onnx_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Introduction to ONNX](intro_onnx.html) \|\| [Exporting a PyTorch model
to ONNX](export_simple_model_to_onnx_tutorial.html) \|\| [Extending the
ONNX exporter operator support](onnx_registry_tutorial.html) \|\|
**\`Export a model with control flow to ONNX**

Export a model with control flow to ONNX
========================================

**Author**: [Xavier Dupré](https://github.com/xadupre)


Overview
========

This tutorial demonstrates how to handle control flow logic while
exporting a PyTorch model to ONNX. It highlights the challenges of
exporting conditional statements directly and provides solutions to
circumvent them.

Conditional logic cannot be exported into ONNX unless they refactored to
use `torch.cond`{.interpreted-text role="func"}. Let\'s start with a
simple model implementing a test.

What you will learn:

-   How to refactor the model to use `torch.cond`{.interpreted-text
    role="func"} for exporting.
-   How to export a model with control flow logic to ONNX.

Prerequisites
-------------

-   `torch >= 2.8`


In [3]:
import torch

Define the Models
=================

Two models are defined:

`ForwardWithControlFlowTest`: A model with a forward method containing
an if-else conditional.

`ModelWithControlFlowTest`: A model that incorporates
`ForwardWithControlFlowTest` as part of a simple MLP. The models are
tested with a random input tensor to confirm they execute as expected.


In [4]:
class ForwardWithControlFlowTest(torch.nn.Module):
    def forward(self, x):
        if x.sum():
            return x * 2
        return -x


class ModelWithControlFlowTest(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(3, 2),
            torch.nn.Linear(2, 1),
            ForwardWithControlFlowTest(),
        )

    def forward(self, x):
        out = self.mlp(x)
        return out


model = ModelWithControlFlowTest()

Exporting the Model: First Attempt
==================================

Exporting this model using torch.export.export fails because the control
flow logic in the forward pass creates a graph break that the exporter
cannot handle. This behavior is expected, as conditional logic not
written using `torch.cond`{.interpreted-text role="func"} is
unsupported.

A try-except block is used to capture the expected failure during the
export process. If the export unexpectedly succeeds, an `AssertionError`
is raised.


In [5]:
x = torch.randn(3)
model(x)

try:
    torch.export.export(model, (x,), strict=False)
    raise AssertionError("This export should failed unless PyTorch now supports this model.")
except Exception as e:
    print(e)

W1001 04:52:20.113000 1170 torch/fx/experimental/symbolic_shapes.py:8255] Unable to find user code corresponding to {u0}


Could not guard on data-dependent expression Eq(u0, 1) (unhinted: Eq(u0, 1)).  (Size-like symbols: none)

consider using data-dependent friendly APIs such as guard_or_false, guard_or_true and statically_known_true.
Caused by: (_export/non_strict_utils.py:1159 in __torch_function__)
For more information, run with TORCH_LOGS="dynamic"
For extended logs when we create symbols, also add TORCHDYNAMO_EXTENDED_DEBUG_CREATE_SYMBOL="u0"
If you suspect the guard was triggered from C++, add TORCHDYNAMO_EXTENDED_DEBUG_CPP=1
For more debugging help, see https://docs.google.com/document/d/1HSuTTVvYH1pTew89Rtpeu84Ht3nQEFTYhAX3Ypa_xJs/edit?usp=sharing

For C++ stack trace, run with TORCHDYNAMO_EXTENDED_DEBUG_CPP=1

The following call raised this error:
  File "/tmp/ipykernel_1170/3521990874.py", line 3, in forward
    if x.sum():


The error above occurred when calling torch.export.export. If you would like to view some more information about this error, and get a list of all other errors that may occ




def forward(self, arg0_1: "f32[2, 3]", arg1_1: "f32[2]", arg2_1: "f32[1, 2]", arg3_1: "f32[1]", arg4_1: "f32[3]"):
    # File: /usr/local/lib/python3.13/dist-packages/torch/nn/modules/linear.py:134 in forward, code: return F.linear(input, self.weight, self.bias)
    linear: "f32[2]" = torch.ops.aten.linear.default(arg4_1, arg0_1, arg1_1);  arg4_1 = arg0_1 = arg1_1 = None
    linear_1: "f32[1]" = torch.ops.aten.linear.default(linear, arg2_1, arg3_1);  linear = arg2_1 = arg3_1 = None
    
    # File: /tmp/ipykernel_1170/3521990874.py:3 in forward, code: if x.sum():
    sum_1: "f32[]" = torch.ops.aten.sum.default(linear_1);  linear_1 = None
    ne: "b8[]" = torch.ops.aten.ne.Scalar(sum_1, 0);  sum_1 = None
    item: "Sym(Eq(u0, 1))" = torch.ops.aten.item.default(ne);  ne = item = None
    



def forward(self, arg0_1: "f32[2, 3]", arg1_1: "f32[2]", arg2_1: "f32[1, 2]", arg3_1: "f32[1]", arg4_1: "f32[3]"):
    # File: /usr/local/lib/python3.13/dist-packages/torch/nn/modules/linear.py:13

Suggested Patch: Refactoring with `torch.cond`{.interpreted-text
role="func"}
\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\-\--

To make the control flow exportable, the tutorial demonstrates replacing
the forward method in `ForwardWithControlFlowTest` with a refactored
version that uses `torch.cond`{.interpreted-text role="func"}\`.

Details of the Refactoring:

Two helper functions (identity2 and neg) represent the branches of the
conditional logic: \* `torch.cond`{.interpreted-text role="func"}[ is
used to specify the condition and the two branches along with the input
arguments. \* The updated forward method is then dynamically assigned to
the ]{.title-ref}[ForwardWithControlFlowTest]{.title-ref}\` instance
within the model. A list of submodules is printed to confirm the
replacement.


In [6]:
def new_forward(x):
    def identity2(x):
        return x * 2

    def neg(x):
        return -x

    return torch.cond(x.sum() > 0, identity2, neg, (x,))


print("the list of submodules")
for name, mod in model.named_modules():
    print(name, type(mod))
    if isinstance(mod, ForwardWithControlFlowTest):
        mod.forward = new_forward

the list of submodules
 <class '__main__.ModelWithControlFlowTest'>
mlp <class 'torch.nn.modules.container.Sequential'>
mlp.0 <class 'torch.nn.modules.linear.Linear'>
mlp.1 <class 'torch.nn.modules.linear.Linear'>
mlp.2 <class '__main__.ForwardWithControlFlowTest'>


Let\'s see what the FX graph looks like.


In [7]:
print(torch.export.export(model, (x,), strict=False))

ExportedProgram:
    class GraphModule(torch.nn.Module):
        def forward(self, p_mlp_0_weight: "f32[2, 3]", p_mlp_0_bias: "f32[2]", p_mlp_1_weight: "f32[1, 2]", p_mlp_1_bias: "f32[1]", x: "f32[3]"):
            # File: /usr/local/lib/python3.13/dist-packages/torch/nn/modules/linear.py:134 in forward, code: return F.linear(input, self.weight, self.bias)
            linear: "f32[2]" = torch.ops.aten.linear.default(x, p_mlp_0_weight, p_mlp_0_bias);  x = p_mlp_0_weight = p_mlp_0_bias = None
            linear_1: "f32[1]" = torch.ops.aten.linear.default(linear, p_mlp_1_weight, p_mlp_1_bias);  linear = p_mlp_1_weight = p_mlp_1_bias = None
    
            # File: /usr/local/lib/python3.13/dist-packages/torch/nn/modules/container.py:253 in forward, code: input = module(input)
            sum_1: "f32[]" = torch.ops.aten.sum.default(linear_1)
            gt: "b8[]" = torch.ops.aten.gt.Scalar(sum_1, 0);  sum_1 = None
    
            # File: <eval_with_key>.5:9 in forward, code: cond = torch

Let\'s export again.


In [9]:
try:
    import onnxscript
except ImportError:
    !pip install onnxscript
    import onnxscript

onnx_program = torch.onnx.export(model, (x,), dynamo=True)
print(onnx_program.model)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 22.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you ha

/tmp/ipykernel_1170/1257855785.py:7: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  onnx_program = torch.onnx.export(model, (x,), dynamo=True)


[torch.onnx] Obtain model graph for `ModelWithControlFlowTest([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ModelWithControlFlowTest([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
<
    ir_version=10,
    opset_imports={'': 20},
    producer_name='pytorch',
    producer_version='2.11.0+cpu',
    domain=None,
    model_version=None,
>
graph(
    name=main_graph,
    inputs=(
        %"x"<FLOAT,[3]>
    ),
    outputs=(
        %"getitem"<FLOAT,[1]>
    ),
    initializers=(
        %"mlp.0.bias"<FLOAT,[2]>{TorchTensor<FLOAT,[2]>(Parameter containing: tensor([-0.5184,  0.1025], requires_grad=True), name='mlp.0.bias')},
        %"mlp.1.bias"<FLOAT,[1]>{TorchTensor<FLOAT,[1]>(Parameter con

/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


Conclusion
==========

This tutorial demonstrates the challenges of exporting models with
conditional logic to ONNX and presents a practical solution using
`torch.cond`{.interpreted-text role="func"}. While the default exporters
may fail or produce imperfect graphs, refactoring the model\'s logic
ensures compatibility and generates a faithful ONNX representation.

By understanding these techniques, we can overcome common pitfalls when
working with control flow in PyTorch models and ensure smooth
integration with ONNX workflows.

Further reading
===============

The list below refers to tutorials that ranges from basic examples to
advanced scenarios, not necessarily in the order they are listed. Feel
free to jump directly to specific topics of your interest or sit tight
and have fun going through all of them to learn all there is about the
ONNX exporter.

::: {.toctree hidden=""}
:::
